# Goodreads Books EDA

This notebook profiles the dataset for search, filtering, semantic-search, recommendation, and graph-readiness decisions. It does not train models, create embeddings, or modify the source data. All derived outputs are written to `data/eda_output/`.

In [5]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import json, math, re
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/raw_data/goodreads_books.parquet'
OUTPUT = ROOT / 'data/eda_output'
OUTPUT.mkdir(exist_ok=True)

# Load only the working dataframe; the original parquet is never overwritten.
df = pd.read_parquet(DATA_PATH)
print(f'Loaded {len(df):,} rows and {len(df.columns):,} columns from {DATA_PATH}')

def present(name):
    return name in df.columns

def is_missing(value):
    if value is None: return True
    if isinstance(value, (list, tuple, set, dict, np.ndarray)): return len(value) == 0
    try:
        if pd.isna(value): return True
    except (TypeError, ValueError): pass
    return isinstance(value, str) and not value.strip()

def values(value):
    if is_missing(value): return []
    if isinstance(value, np.ndarray): return value.tolist()
    return list(value) if isinstance(value, (list, tuple, set)) else [value]

def text(value):
    return '' if is_missing(value) else str(value).strip()

def write_csv(frame, name):
    frame.to_csv(OUTPUT / name, index=False, encoding='utf-8')
    return frame

def write_json(obj, name):
    (OUTPUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

def profile_column(series):
    missing = series.map(is_missing)
    return {'column': series.name, 'dtype': str(series.dtype), 'non_null_count': int((~missing).sum()), 'null_count': int(missing.sum()), 'null_percentage': round(float(missing.mean()*100), 4), 'unique_count': int(series[~missing].astype(str).nunique()), 'unique_percentage': round(float(series[~missing].astype(str).nunique()/max((~missing).sum(),1)*100), 4)}

column_profile = write_csv(pd.DataFrame([profile_column(df[c]) for c in df.columns]), 'column_profile.csv')
missing_values = write_csv(column_profile[['column','null_count','null_percentage']].sort_values('null_percentage', ascending=False), 'missing_values.csv')
display(column_profile)
print('Artifacts directory:', OUTPUT.resolve())

Loaded 205,000 rows and 24 columns from d:\Projects\BookRecSys\data\raw_data\goodreads_books.parquet


,column,dtype,non_null_count,null_count,null_percentage,unique_count,unique_percentage
0,title,str,205000,0,0.0000,198203,96.6844
1,contributors,str,205000,0,0.0000,177153,86.4161
2,first_author,str,204989,11,0.0054,149525,72.9429
3,description,str,178056,26944,13.1434,164802,92.5563
4,genres,str,205000,0,0.0000,47232,23.0400
5,series,str,205000,0,0.0000,21147,10.3156
6,format,str,202150,2850,1.3902,380,0.1880
7,pages,float64,189092,15908,7.7600,1833,0.9694
8,language,str,197141,7859,3.8337,100,0.0507
9,asin,str,203887,1113,0.5429,203887,100.0000


Artifacts directory: D:\Projects\BookRecSys\data\eda_output


## Text, identifiers, ratings, and publication dates

In [6]:
def numeric_stats(column):
    if not present(column): return pd.DataFrame()
    s = pd.to_numeric(df[column], errors='coerce').dropna()
    return pd.DataFrame({'field':[column], 'count':[len(s)], 'missing':[len(df)-len(s)], 'min':[s.min() if len(s) else np.nan], 'max':[s.max() if len(s) else np.nan], 'mean':[s.mean() if len(s) else np.nan], 'median':[s.median() if len(s) else np.nan], 'p25':[s.quantile(.25) if len(s) else np.nan], 'p75':[s.quantile(.75) if len(s) else np.nan], 'p90':[s.quantile(.90) if len(s) else np.nan], 'p95':[s.quantile(.95) if len(s) else np.nan], 'p99':[s.quantile(.99) if len(s) else np.nan]})

if present('title'):
    titles = df['title'].map(text)
    title_stats = pd.DataFrame({'characters': titles.str.len(), 'words': titles.str.split().str.len()}).describe(percentiles=[.25,.5,.75,.9,.95,.99]).T
    write_csv(title_stats.reset_index(names='metric'), 'title_length_stats.csv')
    write_csv(df.assign(_title=titles).query('_title != \"\"').groupby('_title').size().sort_values(ascending=False).head(100).rename('count').reset_index(), 'duplicate_titles.csv')

rating_stats = pd.concat([numeric_stats(c) for c in ['average_rating','ratings_count','reviews_count'] if present(c)], ignore_index=True)
write_json(rating_stats.to_dict('records'), 'rating_stats.json')

date_series = pd.to_datetime(df['publication_date'], errors='coerce') if present('publication_date') else pd.Series(pd.NaT, index=df.index)
date_table = pd.DataFrame({'publication_year': date_series.dt.year, 'publication_month': date_series.dt.month}).value_counts(dropna=False).rename('count').reset_index()
write_csv(date_table, 'publication_year_stats.csv')

id_rows = []
for c in ['book_id','work_id','isbn10','isbn13','asin','url']:
    if present(c):
        s = df[c].map(text); id_rows.append({'field':c, 'missing':int((s=='').sum()), 'unique':int(s[s!=''].nunique()), 'duplicates':int(s[s!=''].duplicated().sum()), 'coverage_pct':round(float((s!='').mean()*100),2)})
write_csv(pd.DataFrame(id_rows), 'identifier_quality.csv')
display(rating_stats)

,field,count,missing,min,max,mean,median,p25,p75,p90,p95,p99
0,average_rating,204998,2,0.0,5.0,2.700206,3.61,0.0,4.0,4.38,4.75,5.00
1,ratings_count,204998,2,0.0,10283153.0,11343.452746,4.00,0.0,54.0,1209.00,7761.30,185709.00
2,reviews_count,204998,2,0.0,277451.0,443.911067,0.00,0.0,6.0,96.00,494.15,9071.06


## Genres, authors, contributors, series, and categorical fields

In [7]:
def explode_values(column, output='value'):
    if not present(column): return pd.DataFrame(columns=[output])
    return pd.DataFrame({output: [item for cell in df[column] for item in values(cell) if not is_missing(item)]})

genre_lists = df['genres'].map(values) if present('genres') else pd.Series([[]]*len(df))
genre_counts = genre_lists.map(len)
genre_long = pd.DataFrame({'genre':[g for row in genre_lists for g in row]}).assign(genre=lambda x: x.genre.map(text))
if len(genre_long):
    genre_stats = genre_long.groupby('genre').size().rename('book_count').reset_index().sort_values('book_count', ascending=False)
    genre_stats['percentage_of_books'] = genre_stats.book_count / len(df) * 100
    if present('average_rating'):
        rating_map = pd.DataFrame({'genre': [g for row in genre_lists for g in row], 'rating': df['average_rating'].repeat(genre_counts).values})
        genre_stats = genre_stats.merge(rating_map.groupby('genre').rating.agg(['mean','median']).rename(columns={'mean':'average_rating_mean','median':'average_rating_median'}), on='genre', how='left')
    write_csv(genre_stats, 'genre_stats.csv')
    pairs = Counter(tuple(sorted(set(row))) for row in genre_lists for _ in [0] if len(set(row)) > 1)
    pair_counts = Counter()
    for row in genre_lists:
        pair_counts.update(combinations(sorted(set(map(text,row))), 2))
    write_csv(pd.DataFrame([{'genre_a':a,'genre_b':b,'book_count':n} for (a,b),n in pair_counts.most_common(100)], columns=['genre_a','genre_b','book_count']), 'genre_cooccurrence.csv')
else: write_csv(pd.DataFrame(), 'genre_stats.csv'); write_csv(pd.DataFrame(), 'genre_cooccurrence.csv')

author_stats = pd.DataFrame()
if present('first_author'):
    author_stats = df.assign(author=df.first_author.map(text)).query('author != \"\"').groupby('author').agg(book_count=('author','size'), mean_rating=('average_rating','mean') if present('average_rating') else ('author','size'), median_rating=('average_rating','median') if present('average_rating') else ('author','size'), total_ratings=('ratings_count','sum') if present('ratings_count') else ('author','size')).reset_index().sort_values('book_count', ascending=False)
write_csv(author_stats.head(100), 'author_stats.csv')

role_rows = []
contributor_rows = []
if present('contributors'):
    for cell in df.contributors:
        for item in values(cell):
            if isinstance(item, dict):
                role_rows.append(item.get('role','')); contributor_rows.append(item)
write_csv(pd.Series(role_rows, name='role').value_counts().rename_axis('role').reset_index(name='count'), 'contributor_role_stats.csv')

series_rows = []
if present('series'):
    for i, cell in enumerate(df.series):
        for item in values(cell):
            if isinstance(item, dict) and text(item.get('name')): series_rows.append({'row':i,'series':text(item.get('name')),'position':item.get('position')})
series_df = pd.DataFrame(series_rows)
if len(series_df):
    series_stats = series_df.groupby('series').row.nunique().rename('number_of_books').reset_index().sort_values('number_of_books', ascending=False)
else: series_stats = pd.DataFrame(columns=['series','number_of_books'])
write_csv(series_stats.head(100), 'series_stats.csv')
for c in ['language','format','publisher']:
    if present(c): write_csv(df[c].map(text).replace('', np.nan).value_counts(dropna=False).rename_axis(c).reset_index(name='count'), f'{c}_stats.csv')

## Readiness, work/edition structure, graph summary, issues, and report

In [8]:
has_desc = df['description'].map(lambda x: not is_missing(x)) if present('description') else pd.Series(False, index=df.index)
has_genre = genre_counts.gt(0)
has_series = df['series'].map(lambda x: len(values(x)) > 0) if present('series') else pd.Series(False, index=df.index)
has_author = df['first_author'].map(lambda x: not is_missing(x)) if present('first_author') else pd.Series(False, index=df.index)
has_language = df['language'].map(lambda x: not is_missing(x)) if present('language') else pd.Series(False, index=df.index)
readiness = []
for field, mask, role in [('title', df.title.map(lambda x: not is_missing(x)) if present('title') else pd.Series(False,index=df.index), 'text'), ('description',has_desc,'text'), ('first_author',has_author,'categorical'), ('genres',has_genre,'relationship'), ('series',has_series,'relationship'), ('language',has_language,'categorical'), ('publisher',df.publisher.map(lambda x:not is_missing(x)) if present('publisher') else pd.Series(False,index=df.index),'categorical'), ('average_rating',df.average_rating.notna() if present('average_rating') else pd.Series(False,index=df.index),'numerical')]:
    readiness.append({'field':field,'coverage_count':int(mask.sum()),'coverage_percentage':round(float(mask.mean()*100),2),'unique_count':int(df[field].map(text).nunique()) if present(field) else 0,'possible_role':role})
search_readiness = write_csv(pd.DataFrame(readiness), 'search_readiness.csv')
semantic = {'description_available':int(has_desc.sum()), 'metadata_fallback':int((~has_desc & (has_genre|has_series|has_author)).sum()), 'low_textual_metadata':int((~has_desc & ~has_genre & ~has_series & ~has_author).sum())}
write_json(semantic, 'semantic_readiness.json')

work_stats = {}
if present('work_id'):
    works = df.assign(_work=df.work_id.map(text)).query('_work != \"\"').groupby('_work').size()
    work_stats = {'unique_works':int(works.size), 'books_per_work':works.describe().to_dict(), 'top_works':works.head(50).to_dict()}
write_json(work_stats, 'work_edition_stats.json')
graph_edges = []
for edge, mask, target in [('Book-WORK-Work',df.work_id.map(lambda x:not is_missing(x)) if present('work_id') else pd.Series(False,index=df.index),'work_id'),('Book-AUTHOR-Author',has_author,'first_author'),('Book-GENRE-Genre',has_genre,'genres'),('Book-SERIES-Series',has_series,'series'),('Book-LANGUAGE-Language',has_language,'language')]: graph_edges.append({'edge_type':edge,'number_of_edges':int(mask.sum()),'coverage':round(float(mask.mean()*100),2),'target_field':target})
graph_readiness = {'nodes':['Book','Work','Author','Genre','Series','Publisher','Language'],'edges':graph_edges}
write_json(graph_readiness, 'graph_readiness.json')

issues = []
for field, mask in [('description_missing',~has_desc),('genres_missing',~has_genre),('series_missing',~has_series),('author_missing',~has_author),('language_missing',~has_language)]: issues.append({'issue':field,'affected_rows':int(mask.sum()),'percentage':round(float(mask.mean()*100),2),'example_values':[]})
if present('publication_date'): issues.append({'issue':'invalid_publication_date','affected_rows':int(date_series.isna().sum()),'percentage':round(float(date_series.isna().mean()*100),2),'example_values':df.loc[date_series.isna(),'publication_date'].dropna().astype(str).head(5).tolist()})
write_csv(pd.DataFrame(issues), 'data_quality_issues.csv')

summary = {'dataset':{'rows':len(df),'columns':len(df.columns),'column_names':df.columns.tolist()}, 'description':{'available':int(has_desc.sum()),'missing':int((~has_desc).sum()),'coverage_pct':round(float(has_desc.mean()*100),2)}, 'genres':{'unique_genres':int(genre_long.genre.nunique()) if len(genre_long) else 0,'books_with_genres':int(has_genre.sum()),'top_genres':genre_stats.head(20).to_dict('records') if len(genre_long) else []}, 'authors':{'unique_authors':int(df.first_author.map(text).replace('',np.nan).nunique()) if present('first_author') else 0}, 'series':{'unique_series':int(series_df.series.nunique()) if len(series_df) else 0,'books_in_series':int(has_series.sum())}, 'ratings':rating_stats.to_dict('records'), 'works_and_editions':work_stats, 'semantic_search_readiness':semantic, 'graph_readiness':graph_readiness, 'search_readiness':readiness, 'data_quality':issues}
write_json(summary, 'eda_summary.json')
report = f'''# Goodreads Dataset EDA

## 1. Dataset Overview
- Rows: {len(df):,}
- Columns: {len(df.columns):,}

## 2. Missing Data
- Description available: {int(has_desc.sum()):,} ({has_desc.mean()*100:.2f}%).

## 3. Description
Description coverage and metadata fallback groups are saved in `semantic_readiness.json`.

## 4. Genres
- Unique genres: {int(genre_long.genre.nunique()) if len(genre_long) else 0:,}.

## 5. Authors
- Unique first authors: {summary['authors']['unique_authors']:,}.

## 6. Series
- Books in series: {int(has_series.sum()):,}.

## 7. Language
See `language_stats.csv`.

## 8. Format
See `format_stats.csv`.

## 9. Publisher
See `publisher_stats.csv`.

## 10. Publication Date
See `publication_year_stats.csv`.

## 11. Ratings and Popularity
See `rating_stats.json`.

## 12. Works and Editions
See `work_edition_stats.json`.

## 13. Search Readiness
See `search_readiness.csv`.

## 14. Semantic Search Readiness
See `semantic_readiness.json`.

## 15. Graph Readiness
See `graph_readiness.json`.

## 16. Data Quality Issues
See `data_quality_issues.csv`.

## 17. Important Observations
- These are descriptive statistics only; no causal conclusions or model choices are made.
- Original dataset values are not normalized or overwritten.
'''
(OUTPUT / 'eda_report.md').write_text(report, encoding='utf-8')
required = ['eda_summary.json','column_profile.csv','missing_values.csv','genre_stats.csv','genre_cooccurrence.csv','author_stats.csv','contributor_role_stats.csv','series_stats.csv','language_stats.csv','format_stats.csv','publisher_stats.csv','publication_year_stats.csv','rating_stats.json','work_edition_stats.json','graph_readiness.json','semantic_readiness.json','search_readiness.csv','data_quality_issues.csv','eda_report.md']
missing_artifacts = [name for name in required if not (OUTPUT/name).exists()]
print('EDA COMPLETED')
print(f'Rows analyzed: {len(df):,}')
print(f'Columns analyzed: {len(df.columns):,}')
print('Artifacts saved to:', OUTPUT.resolve())
print('Missing artifacts:', missing_artifacts or 'none')

EDA COMPLETED
Rows analyzed: 205,000
Columns analyzed: 24
Artifacts saved to: D:\Projects\BookRecSys\data\eda_output
Missing artifacts: none
